<a href="https://colab.research.google.com/github/hermannvargens/absorcao/blob/main/Absor%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Dimensionamento Preliminar do Diâmetro de uma Torre de Absorção Recheada para Recuperação de Vapores de Benzeno**

### Contexto do Processo

Em uma unidade industrial, uma corrente de ar residual contaminada com vapores de benzeno ($\text{C}_6\text{H}_6$) precisa ser tratada antes do descarte na atmosfera, visando tanto o controle de emissões poluentes quanto a recuperação do solvente. Para realizar essa operação, projeta-se uma torre de absorção operando em contracorrente.

O ar contaminado entra pela base da coluna e sobe através de um leito recheado, enquanto um óleo absorvedor pesado e não volátil — composto predominantemente por tetradecano ($\text{C}_{14}\text{H}_{30}$) — é alimentado no topo e escorre por gravidade, promovendo a transferência de massa do benzeno da fase gasosa para a fase líquida.

Para assegurar uma boa capacidade de escoamento com baixa perda de carga, a coluna será preenchida com **anéis Hiflow metálicos de 50 mm**. O critério de dimensionamento hidráulico estipula que a coluna deve operar com uma perda de carga na fase gasosa fixada em **$300\text{ Pa/m}$** de altura de leito recheado.

---

### Condições Críticas de Operação (Fundo da Coluna)

O diâmetro interno da torre cilíndrica deve ser dimensionado para suportar as condições hidrodinâmicas mais severas, que ocorrem no fundo do absorvedor, onde as vazões de vapor e de líquido atingem seus valores máximos:

* **Fase gasosa (corrente de alimentação na entrada inferior):**
* Mistura: Ar atmosférico com vapores de benzeno ($y_{\text{benzeno}} = 0{,}074$ fração molar)
* Vazão mássica ($V'$): $1{,}325\text{ kg/s}$
* Massa específica ($\rho_G$): $1{,}325\text{ kg/m}^3$


* **Fase líquida (solvente enriquecido na saída de fundo):**
* Solução: Benzeno absorvido em tetradecano ($x_{\text{benzeno}} = 0{,}324$ fração molar)
* Vazão mássica ($L'$): $1{,}413\text{ kg/s}$
* Massa específica ($\rho_L$): $780\text{ kg/m}^3$
* Viscosidade dinâmica ($\mu_L$): $1{,}41\text{ cP}$ ($1{,}41 \times 10^{-3}\text{ Pa}\cdot\text{s}$)


* **Características do recheio selecionado:**
* Tipo: Anéis Hiflow metálicos de 50 mm
* Fator de empacotamento (*packing factor*, $F_p$): $16\text{ ft}^{-1}$ (ou $16\text{ ft}^2/\text{ft}^3$)


### Método de Billet & Schultes

In [1]:
# ==============================================================================
# PROJETO DE UMA TORRE RECHEADA - MÉTODO DE BILLET & SCHULTES
# Este programa calcula o diâmetro (D) necessário para atender a uma perda de carga
# especificada na torre, reproduzindo o algoritmo do livro.
# ==============================================================================

import math
from scipy.optimize import fsolve

# ------------------------------------------------------------------------------
# 1. DADOS DE ENTRADA (PROPRIEDADES DOS FLUIDOS E CRITÉRIOS DE PROJETO)
# ------------------------------------------------------------------------------
# Líquido (fundo da torre)
mL = 0.804           # Vazão mássica do líquido [kg/s]
rho_L = 986.0        # Densidade do líquido [kg/m³]
mu_L = 0.000631      # Viscosidade dinâmica do líquido [Pa·s]

# Gás (fundo da torre)
mG = 2.202           # Vazão mássica do gás [kg/s]
rho_G = 1.923        # Densidade do gás [kg/m³]
mu_G = 1.45e-5       # Viscosidade dinâmica do gás [Pa·s]

# Critério de projeto e constantes físicas
delta_P_alvo = 300.0 # Perda de carga desejada [Pa/m]
g = 9.8              # Aceleração da gravidade [m/s²]

# ------------------------------------------------------------------------------
# 2. DADOS DO RECHEIO (Anéis Hiflow metálicos de 50 mm)
# ------------------------------------------------------------------------------
Fp = 16.0            # Fator de empacotamento [ft²/ft³]
a = 92.3             # Área superficial específica do recheio [m²/m³]
epsilon = 0.977      # Fração de vazios (porosidade) [adimensional]
Ch = 0.876           # Constante empírica para retenção de líquido
Cp = 0.421           # Constante empírica para perda de carga

# Diâmetro hidráulico efetivo da partícula de recheio [m]
dp = 6.0 * (1.0 - epsilon) / a

# Vazões volumétricas de trabalho [m³/s]
QL = mL / rho_L
QG = mG / rho_G

# ------------------------------------------------------------------------------
# 3. ESTIMATIVA INICIAL DO DIÂMETRO (CHUTE INICIAL VIA INUNDAÇÃO / FLOODING)
# ------------------------------------------------------------------------------
# Fator de conversão de unidades do Fp para o cálculo empírico de inundação
Fpu = Fp * 1.0

# Parâmetro de fluxo (X)
X = (mL / mG) * math.sqrt(rho_G / rho_L)

# Capacidade na inundação (Yflood)
Yflood = math.exp(-(3.5021 + 1.028 * math.log(X) + 0.11093 * (math.log(X))**2))

# Fator C capacidade na inundação (Csf) e velocidade de gás na inundação (vGf)
Csf = math.sqrt(Yflood / (Fpu * (mu_L ** 0.1)))
vGf = Csf / math.sqrt(rho_G / (rho_L - rho_G))

# Chute inicial adotando 70% da velocidade de inundação (f = 0.7)
f = 0.7
D_inicial = math.sqrt((4.0 * QG) / (f * vGf * math.pi))

# ------------------------------------------------------------------------------
# 4. FUNÇÃO HIDRÁULICA DO MODELO DE BILLET & SCHULTES
# ------------------------------------------------------------------------------
def calcular_diferenca_delta_P(D):
    """
    Esta função recebe um diâmetro D, calcula a perda de carga que ocorreria
    na torre e retorna a diferença em relação à meta de 300 Pa/m.
    O método iterativo busca o valor de D onde essa diferença é ZERO.
    """
    if D <= 0:
        return 1e6

    # 4.1 Hidráulica da fase líquida
    area_torre = (math.pi * (D ** 2)) / 4.0
    vL = QL / area_torre
    ReL = (vL * rho_L) / (a * mu_L)
    FrL = ((vL ** 2) * a) / g

    # Área específica molhada (ah)
    if ReL >= 5.0:
        ah = a * 0.85 * Ch * (ReL ** 0.25) * (FrL ** 0.1)
    else:
        ah = a * Ch * (ReL ** 0.25) * (FrL ** 0.1)

    # Retenção de líquido no leito (Liquid Holdup, hL)
    hL = ((12.0 * FrL / ReL) ** (1.0 / 3.0)) * ((ah / a) ** (2.0 / 3.0))

    # 4.2 Efeito de parede e hidráulica do gás
    KW = 1.0 / (1.0 + (2.0 * dp) / (3.0 * D * (1.0 - epsilon)))
    vG = QG / area_torre
    ReG = (vG * dp * rho_G * KW) / ((1.0 - epsilon) * mu_G)

    # Coeficiente de atrito no leito seco (psi_0)
    psi_0 = Cp * ((64.0 / ReG) + (1.8 / (ReG ** 0.08)))

    # Perda de carga no leito seco (Delta P0) [Pa/m]
    delta_P0 = (psi_0 * a * rho_G * (vG ** 2)) / (2.0 * (epsilon ** 3) * KW)

    # 4.3 Perda de carga com o leito irrigado (Delta P) [Pa/m]
    if (epsilon - hL) <= 0:
        return 1e6

    delta_P_calculado = delta_P0 * ((epsilon / (epsilon - hL)) ** 1.5) * math.exp(ReL / 200.0)

    return delta_P_calculado - delta_P_alvo

# ------------------------------------------------------------------------------
# 5. RESOLUÇÃO ITERATIVA E CÁLCULO DAS VARIÁVEIS FINAIS
# ------------------------------------------------------------------------------
# Resolução do diâmetro final
diametro_final = fsolve(calcular_diferenca_delta_P, x0=D_inicial)[0]

# Área transversal final da coluna e velocidade real de operação do gás
area_transversal = (math.pi * (diametro_final ** 2)) / 4.0
vG_operacao = QG / area_transversal
fracao_flooding = (vG_operacao / vGf) * 100.0

# ------------------------------------------------------------------------------
# 6. EXIBIÇÃO DOS RESULTADOS CONSOLIDADOS
# ------------------------------------------------------------------------------
print("=" * 65)
print("             RESULTADOS DO DIMENSIONAMENTO HIDRÁULICO            ")
print("=" * 65)
print(f"1. Parâmetro de fluxo (X)             : {X:.5f}")
print(f"2. Capacidade de inundação (Y_flood)  : {Yflood:.5f}")
print(f"3. Velocidade de inundação (vGf)      : {vGf:.3f} m/s")
print(f"4. Velocidade de operação admitida    : {vG_operacao:.3f} m/s ({fracao_flooding:.1f}% do flooding)")
print(f"5. Área transversal da coluna (A)     : {area_transversal:.4f} m²")
print(f"6. Diâmetro calculado da coluna (D)   : {diametro_final:.3f} m ({diametro_final * 100:.1f} cm)")
print("=" * 65)


             RESULTADOS DO DIMENSIONAMENTO HIDRÁULICO            
1. Parâmetro de fluxo (X)             : 0.01612
2. Capacidade de inundação (Y_flood)  : 0.31699
3. Velocidade de inundação (vGf)      : 4.602 m/s
4. Velocidade de operação admitida    : 2.685 m/s (58.3% do flooding)
5. Área transversal da coluna (A)     : 0.4264 m²
6. Diâmetro calculado da coluna (D)   : 0.737 m (73.7 cm)


/tmp/ipykernel_727/3654529628.py:105: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  delta_P_calculado = delta_P0 * ((epsilon / (epsilon - hL)) ** 1.5) * math.exp(ReL / 200.0)


# Correlação Generalizada de Inundação (GPDC / Eckert)

In [27]:
# ==============================================================================
# DIMENSIONAMENTO DE TORRE RECHEADA - MODELO DE KISTER & GILL (1991)
# Com verificação da Porcentagem de Inundação (% Flooding)
# ==============================================================================

import math

# ------------------------------------------------------------------------------
# 1. DADOS DE ENTRADA DO PROCESSO (SISTEMA INTERNACIONAL)
# ------------------------------------------------------------------------------
mL = 0.804           # Vazão mássica do líquido [kg/s][cite: 1]
rho_L = 986.0        # Densidade do líquido [kg/m³][cite: 1]
mu_L_Pa_s = 0.000631 # Viscosidade dinâmica do líquido [Pa·s][cite: 1]

mG = 2.202           # Vazão mássica do gás [kg/s][cite: 1]
rho_G = 1.923        # Densidade do gás [kg/m³][cite: 1]

Fp = 16.0            # Fator de empacotamento [ft⁻¹][cite: 1]
delta_P_alvo = 300.0 # Perda de carga de projeto [Pa/m][cite: 1]

# Vazão volumétrica do gás
QG = mG / rho_G      # [m³/s]

# ------------------------------------------------------------------------------
# 2. CONVERSÕES DE UNIDADES E PARÂMETRO DE FLUXO
# ------------------------------------------------------------------------------
# 1 Pa/m = 0.0012232 in H2O/ft
delta_P_inH2O = delta_P_alvo * 0.0012232

# Viscosidade cinemática em centistokes [cSt]
nu_L_cSt = (mu_L_Pa_s / rho_L) * 1e6

# Parâmetro de fluxo (FLV ou X)
F_LV = (mL / mG) * math.sqrt(rho_G / rho_L)

# ------------------------------------------------------------------------------
# 3. VELOCIDADE DE INUNDAÇÃO DE REFERÊNCIA (GPDC FLOODING)
# ------------------------------------------------------------------------------
# Equação da capacidade no ponto de afogamento (Flooding)
Y_flood = math.exp(-(3.5021 + 1.028 * math.log(F_LV) + 0.11093 * (math.log(F_LV))**2))
Csf = math.sqrt(Y_flood / (Fp * (mu_L_Pa_s ** 0.1)))
vGf = Csf / math.sqrt(rho_G / (rho_L - rho_G))

# ------------------------------------------------------------------------------
# 4. DIMENSIONAMENTO VIA MODELO DE KISTER & GILL (1991)
# ------------------------------------------------------------------------------
# Equação: ΔP = 0.115 * Fp * [ Cs * nu^0.05 ]^2 * 10^(1.05 * F_LV)
termo_denominador = 0.115 * Fp * (10.0 ** (1.05 * F_LV))
Cs_eng = math.sqrt(delta_P_inH2O / termo_denominador) / (nu_L_cSt ** 0.05)

# Conversão de Cs: ft/s -> m/s
Cs_SI = Cs_eng * 0.3048

# Velocidade superficial do gás uG [m/s]
uG = Cs_SI * math.sqrt((rho_L - rho_G) / rho_G)

# Área transversal [m²] e Diâmetro [m]
area_torre = QG / uG
diametro = math.sqrt((4.0 * area_torre) / math.pi)

# Porcentagem de inundação operacional
percentual_flooding = (uG / vGf) * 100.0

# ------------------------------------------------------------------------------
# 5. EXIBIÇÃO DOS RESULTADOS
# ------------------------------------------------------------------------------
print("=" * 70)
print("     DIMENSIONAMENTO VIA KISTER & GILL COM ANÁLISE DE INUNDAÇÃO")
print("=" * 70)
print(f"• Parâmetro de Fluxo (F_LV)          : {F_LV:.5f}")
print(f"• Perda de Carga de Projeto          : {delta_P_alvo:.1f} Pa/m")
print("-" * 70)
print(f"• Velocidade Superficial do Gás (uG) : {uG:.3f} m/s")
print(f"• Velocidade de Inundação (vGf)      : {vGf:.3f} m/s")
print(f"• Porcentagem de Inundação Operacional: {percentual_flooding:.1f} % do Flooding")
print("-" * 70)
print(f"• Área da Seção Transversal (A)      : {area_torre:.4f} m²")
print(f"• Diâmetro Interno da Coluna (D)     : {diametro:.3f} m  ({diametro * 100:.1f} cm)")
print("=" * 70)

# Diagnóstico de projeto
if 50.0 <= percentual_flooding <= 70.0:
    print("STATUS: Projeto seguro e dentro da faixa industrial típica (50% a 70% do flooding).")
elif percentual_flooding > 70.0:
    print("STATUS: Alerta - Operação muito próxima da inundação (risco operacional).")
else:
    print("STATUS: Coluna operando com folga excessiva (risco de má distribuição de líquido).")

     DIMENSIONAMENTO VIA KISTER & GILL COM ANÁLISE DE INUNDAÇÃO
• Parâmetro de Fluxo (F_LV)          : 0.01612
• Perda de Carga de Projeto          : 300.0 Pa/m
----------------------------------------------------------------------
• Velocidade Superficial do Gás (uG) : 3.088 m/s
• Velocidade de Inundação (vGf)      : 4.602 m/s
• Porcentagem de Inundação Operacional: 67.1 % do Flooding
----------------------------------------------------------------------
• Área da Seção Transversal (A)      : 0.3708 m²
• Diâmetro Interno da Coluna (D)     : 0.687 m  (68.7 cm)
STATUS: Projeto seguro e dentro da faixa industrial típica (50% a 70% do flooding).


# Gráfico

In [11]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

# ==============================================================================
# BASE DE DADOS DOS RECHEIOS (Parâmetros de Billet & Schultes e GPDC)
# ==============================================================================
BANCO_RECHEIOS = {
    'Anéis Hiflow 50 mm (Metálico)': {
        'Fp': 16.0, 'a': 92.3, 'epsilon': 0.977, 'Ch': 0.876, 'Cp': 0.421
    },
    'Anéis Pall 50 mm (Metálico)': {
        'Fp': 20.0, 'a': 110.0, 'epsilon': 0.950, 'Ch': 1.050, 'Cp': 0.480
    },
    'Anéis Pall 25 mm (Metálico)': {
        'Fp': 56.0, 'a': 215.0, 'epsilon': 0.940, 'Ch': 1.150, 'Cp': 0.520
    },
    'Anéis Raschig 50 mm (Cerâmico)': {
        'Fp': 65.0, 'a': 112.0, 'epsilon': 0.740, 'Ch': 1.400, 'Cp': 0.750
    },
    'Anéis Raschig 25 mm (Cerâmico)': {
        'Fp': 155.0, 'a': 190.0, 'epsilon': 0.730, 'Ch': 1.550, 'Cp': 0.850
    }
}

# ------------------------------------------------------------------------------
# PROPRIEDADES FÍSICAS DOS FLUIDOS E CONSTANTES (EXEMPLO 5.3)
# ------------------------------------------------------------------------------
rho_L = 986.0        # Massa específica do líquido [kg/m³]
mu_L = 0.000631      # Viscosidade dinâmica do líquido [Pa·s]

rho_G = 1.923        # Massa específica do gás [kg/m³]
mu_G = 1.45e-5       # Viscosidade dinâmica do gás [Pa·s]
g = 9.8              # Aceleração da gravidade [m/s²]

# ==============================================================================
# FUNÇÃO HIDRÁULICA DE BILLET & SCHULTES
# ==============================================================================
def calcular_delta_P_billet(D, mG_val, mL_val, a_rec, eps_rec, Ch_rec, Cp_rec):
    """Calcula a perda de carga em Pa/m segundo Billet & Schultes."""
    if D <= 0:
        return 1e6

    area = (np.pi * (D ** 2)) / 4.0
    dp = 6.0 * (1.0 - eps_rec) / a_rec

    QG_local = mG_val / rho_G
    QL_local = mL_val / rho_L

    # Fase Líquida
    vL = QL_local / area
    ReL = (vL * rho_L) / (a_rec * mu_L)
    FrL = ((vL ** 2) * a_rec) / g

    if ReL >= 5.0:
        ah = a_rec * 0.85 * Ch_rec * (ReL ** 0.25) * (FrL ** 0.1)
    else:
        ah = a_rec * Ch_rec * (ReL ** 0.25) * (FrL ** 0.1)

    hL = ((12.0 * FrL / ReL) ** (1.0 / 3.0)) * ((ah / a_rec) ** (2.0 / 3.0))

    # Fase Gasosa
    KW = 1.0 / (1.0 + (2.0 * dp) / (3.0 * D * (1.0 - eps_rec)))
    vG = QG_local / area
    ReG = (vG * dp * rho_G * KW) / ((1.0 - eps_rec) * mu_G)

    psi_0 = Cp_rec * ((64.0 / ReG) + (1.8 / (ReG ** 0.08)))
    delta_P0 = (psi_0 * a_rec * rho_G * (vG ** 2)) / (2.0 * (eps_rec ** 3) * KW)

    if (eps_rec - hL) <= 0.001:
        return 2000.0  # Afogamento completo

    delta_P = delta_P0 * ((eps_rec / (eps_rec - hL)) ** 1.5) * np.exp(ReL / 200.0)
    return min(float(delta_P), 2000.0)

# ==============================================================================
# ATUALIZAÇÃO DOS GRÁFICOS E PAINEL
# ==============================================================================
def atualizar_painel(recheio_selecionado, mG_val, mL_val, fracao_flood, diametro_escolhido):
    dados_rec = BANCO_RECHEIOS[recheio_selecionado]
    Fp = dados_rec['Fp']
    a_rec = dados_rec['a']
    eps_rec = dados_rec['epsilon']
    Ch_rec = dados_rec['Ch']
    Cp_rec = dados_rec['Cp']

    # Vazões volumétricas atuais
    QG = mG_val / rho_G
    QL = mL_val / rho_L

    # Parâmetro de fluxo adimensional (GPDC)
    X_atual = (mL_val / mG_val) * np.sqrt(rho_G / rho_L)

    # --------------------------------------------------------------------------
    # 1. CÁLCULOS DE INUNDAÇÃO (FLOODING)
    # --------------------------------------------------------------------------
    X_faixa = np.logspace(-2.5, 0.5, 200)
    Y_flood_faixa = np.exp(-(3.5021 + 1.028 * np.log(X_faixa) + 0.11093 * (np.log(X_faixa)**2)))
    Y_70_faixa = (0.70 ** 2) * Y_flood_faixa
    Y_50_faixa = (0.50 ** 2) * Y_flood_faixa

    # Ponto no Flooding para este sistema
    Y_flood_op = float(np.exp(-(3.5021 + 1.028 * np.log(X_atual) + 0.11093 * (np.log(X_atual)**2))))
    Csf = np.sqrt(Y_flood_op / (Fp * (mu_L ** 0.1)))
    vGf = Csf / np.sqrt(rho_G / (rho_L - rho_G))

    # Diâmetro mínimo no Flooding (100%)
    D_flood = np.sqrt((4.0 * QG) / (vGf * np.pi))

    # Diâmetro projetado com a fração de inundação do controle
    D_projetado = np.sqrt((4.0 * QG) / (fracao_flood * vGf * np.pi))
    dP_projetado = calcular_delta_P_billet(D_projetado, mG_val, mL_val, a_rec, eps_rec, Ch_rec, Cp_rec)

    # Estado no Diâmetro Selecionado manualmente
    area_escolhida = (np.pi * (diametro_escolhido ** 2)) / 4.0
    vG_op = QG / area_escolhida
    percentual_flood_real = (vG_op / vGf) * 100.0

    Cs_op = vG_op * np.sqrt(rho_G / (rho_L - rho_G))
    Y_op = (Cs_op ** 2) * Fp * (mu_L ** 0.1)

    # Perda de carga real exata para o diâmetro escolhido
    dP_op = calcular_delta_P_billet(diametro_escolhido, mG_val, mL_val, a_rec, eps_rec, Ch_rec, Cp_rec)

    # --------------------------------------------------------------------------
    # 2. CURVA HIDRÁULICA NO INTERVALO DE DIÂMETROS
    # --------------------------------------------------------------------------
    D_min_plot = max(0.35, D_flood * 0.95)
    D_max_plot = max(1.40, D_projetado * 1.35)
    D_faixa = np.linspace(D_min_plot, D_max_plot, 90)
    dP_faixa = [calcular_delta_P_billet(d, mG_val, mL_val, a_rec, eps_rec, Ch_rec, Cp_rec) for d in D_faixa]

    # --------------------------------------------------------------------------
    # 3. CONSTRUÇÃO DOS GRÁFICOS
    # --------------------------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.8))

    # --- GRÁFICO 1: GPDC ---
    ax1.loglog(X_faixa, Y_flood_faixa, 'r-', lw=2.2, label='100% Inundação (Flooding)')
    ax1.loglog(X_faixa, Y_70_faixa, 'b--', lw=1.8, label='70% da Inundação')
    ax1.loglog(X_faixa, Y_50_faixa, 'g:', lw=1.8, label='50% da Inundação')

    ax1.plot(X_atual, Y_op, 'o', color='purple', markersize=9, zorder=5,
             label=f'Operação Selecionada ({percentual_flood_real:.1f}% Flood)')
    ax1.axvline(X_atual, color='gray', linestyle=':', alpha=0.6)

    ax1.set_title('Diagrama GPDC - Curvas de Capacidade', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Parâmetro de Fluxo $X = (m_L / m_G) \\cdot (\\rho_G / \\rho_L)^{0.5}$', fontsize=11)
    ax1.set_ylabel('Parâmetro de Capacidade $Y$', fontsize=11)
    ax1.set_xlim(0.005, 1.0)
    ax1.set_ylim(0.001, 1.0)
    ax1.grid(True, which="both", ls="--", alpha=0.35)
    ax1.legend(loc='lower left', frameon=True, fontsize=9.5)

    # --- GRÁFICO 2: JANELA OPERACIONAL ---
    ax2.plot(D_faixa, dP_faixa, color='#1f77b4', lw=2.5, label='Curva Billet & Schultes $\\Delta P(D)$')

    # Faixa recomendada de absorção
    ax2.axhspan(200, 400, color='green', alpha=0.18, label='Faixa Recomendada (200 - 400 Pa/m)')
    ax2.axhline(300, color='darkgreen', linestyle='--', lw=1.2, label='Alvo de Projeto (300 Pa/m)')

    # Limite físico de inundação
    ax2.axvline(D_flood, color='red', linestyle='--', lw=1.5, label=f'Flooding 100% ($D = {D_flood:.3f}$ m)')

    # Linha dinâmica controlada pelo slider de fração de inundação
    ax2.axvline(D_projetado, color='orange', linestyle='-.', lw=2.0,
                label=f'Projeto f={fracao_flood*100:.0f}% ($D = {D_projetado:.3f}$ m)')

    # Ponto operacional selecionado
    cor_ponto = 'purple' if 200 <= dP_op <= 400 else ('red' if dP_op > 600 else 'brown')
    ax2.plot(diametro_escolhido, min(dP_op, 1150), 'o', color=cor_ponto, markersize=10, zorder=5,
             label=f'D Selecionado: {diametro_escolhido:.3f} m ($\\Delta P = {dP_op:.0f}$ Pa/m)')

    ax2.set_title('Janela Hidráulica Operacional', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Diâmetro Interno da Coluna $D$ [m]', fontsize=11)
    ax2.set_ylabel('Perda de Carga do Gás $\\Delta P / Z$ [Pa/m]', fontsize=11)
    ax2.set_xlim(D_min_plot, D_max_plot)
    ax2.set_ylim(0, 1200)
    ax2.grid(True, ls=":", alpha=0.5)
    ax2.legend(loc='upper right', frameon=True, fontsize=9)

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------------------------
    # 4. PAINEL DE LEITURA
    # --------------------------------------------------------------------------
    print("=" * 80)
    print(f" RECHEIO: {recheio_selecionado} (Fp = {Fp} ft⁻¹)")
    print(f" VAZÕES : Gás (mG) = {mG_val:.3f} kg/s  |  Líquido (mL) = {mL_val:.3f} kg/s")
    print("-" * 80)
    print(f" • Parâmetro de Fluxo (X)                  : {X_atual:.5f}")
    print(f" • Velocidade de Inundação (vGf)           : {vGf:.3f} m/s")
    print(f" • Diâmetro de Inundação Total (100% Flood): {D_flood:.3f} m")
    print(f" • Diâmetro Projetado para f = {fracao_flood*100:.0f}%       : {D_projetado:.3f} m  (ΔP ≈ {dP_projetado:.1f} Pa/m)")
    print(f" • Diâmetro Selecionado em Operação        : {diametro_escolhido:.3f} m")
    print(f" • Velocidade Superficial do Gás (vG)      : {vG_op:.3f} m/s ({percentual_flood_real:.1f}% do Flood)")
    print(f" • Perda de Carga Real (Billet & Schultes) : {dP_op:.1f} Pa/m")

    if dP_op >= 600 or percentual_flood_real >= 90:
        print("\n [ALERTA]: Coluna em regime de AFOGAMENTO ou com perda de carga excessiva!")
    elif 200 <= dP_op <= 400:
        print("\n [DIAGNÓSTICO]: Operação dentro da FAIXA ÓTIMA de projeto (200 a 400 Pa/m).")
    else:
        print("\n [DIAGNÓSTICO]: Coluna fora da faixa recomendada (superdimensionada ou sobrecarregada).")
    print("=" * 80)

# ==============================================================================
# CONTROLES INTERATIVOS (WIDGETS)
# ==============================================================================
widget_recheio = widgets.Dropdown(
    options=list(BANCO_RECHEIOS.keys()),
    value='Anéis Hiflow 50 mm (Metálico)',
    description='Recheio:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

widget_mG = widgets.FloatSlider(
    value=2.202,
    min=0.50,
    max=5.00,
    step=0.05,
    description='Vazão Gás mG [kg/s]:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

widget_mL = widgets.FloatSlider(
    value=0.804,
    min=0.10,
    max=3.00,
    step=0.05,
    description='Vazão Líquido mL [kg/s]:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

widget_flood = widgets.FloatSlider(
    value=0.70,
    min=0.40,
    max=0.85,
    step=0.01,
    description='Fração Flood (f):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

widget_diametro = widgets.FloatSlider(
    value=0.737,
    min=0.40,
    max=1.50,
    step=0.005,
    description='Diâmetro Coluna D [m]:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

# Organização visual em duas linhas de controles
ui = widgets.VBox([
    widgets.HTML("<h3>Painel Interativo de Hidráulica e Dimensionamento de Torres Recheadas</h3>"),
    widgets.HBox([widget_recheio, widget_mG, widget_mL]),
    widgets.HBox([widget_flood, widget_diametro])
])

saida_interativa = widgets.interactive_output(
    atualizar_painel,
    {
        'recheio_selecionado': widget_recheio,
        'mG_val': widget_mG,
        'mL_val': widget_mL,
        'fracao_flood': widget_flood,
        'diametro_escolhido': widget_diametro
    }
)

display(ui, saida_interativa)


Output()

# Perguntas

---

Aqui está um questionário didático estruturado com base nas interações e fenômenos observados no simulador, acompanhado das respostas conceituais e quantitativas esperadas para orientar o debate em sala de aula:

---

### 1. Influência da geometria do recheio na capacidade da coluna

**Pergunta:**

Mantendo todas as vazões e o diâmetro da coluna fixos ($D = 0{,}737\text{ m}$), altere o recheio de *Anéis Hiflow 50 mm* para *Anéis Pall 25 mm*. O que acontece com a perda de carga e com a condição operacional da coluna? Explique o porquê do ponto de vista físico.

**Resposta esperada:**

A perda de carga dispara para valores excessivos e a coluna entra em regime de afogamento (*flooding*) total.

*Justificativa física:* Os anéis Raschig de 25 mm possuem uma área específica muito maior ($a = 190\text{ m}^2/\text{m}^3$) e uma fração de vazios significativamente menor ($\varepsilon = 0{,}73$ contra $\varepsilon = 0{,}977$ dos anéis Hiflow), o que se reflete em um fator de empacotamento muito mais alto ($F_p = 155\text{ ft}^{-1}$ contra $16\text{ ft}^{-1}$). Menor porosidade significa canais de passagem de vapor mais estreitos; com mais atrito e menor área livre de escoamento, a retenção de líquido ($h_L$) sobe rapidamente, estrangulando o fluxo de gás.

---

### 2. Efeito do diâmetro na velocidade superficial e na perda de carga

**Pergunta:**

No gráfico da direita (*Janela Hidráulica Operacional*), por que a curva de perda de carga ($\Delta P$) não decresce linearmente à medida que aumentamos o diâmetro ($D$), mas sim com um decaimento abrupto?

**Resposta esperada:**

A velocidade superficial do gás é inversamente proporcional à área transversal, ou seja, inversamente proporcional ao quadrado do diâmetro ($v_G \propto 1/D^2$). Por sua vez, a perda de carga no leito seco e no leito irrigado depende diretamente da energia cinética do vapor, que varia com o quadrado da velocidade ($v_G^2$). Logo, em primeira aproximação, $\Delta P \propto 1/D^4$. Além disso, nos diâmetros menores, a retenção de líquido ($h_L$) restringe ainda mais o espaço vazio, provocando uma elevação assintótica da perda de carga próxima ao afogamento.

---

### 3. Variação da vazão de gás ($m_G$) no diagrama GPDC

**Pergunta:**

Ao aumentar a vazão de vapor ($m_G$) no controle deslizante, para onde o ponto operacional se desloca no diagrama GPDC (gráfico da esquerda)? O que esse deslocamento representa em termos de estabilidade da torre?

**Resposta esperada:**

O parâmetro de fluxo é dado por $X = (m_L / m_G) \sqrt{\rho_G / \rho_L}$. Portanto, ao aumentar $m_G$, o valor de $X$ diminui, deslocando o ponto operacional para a **esquerda**. Simultaneamente, a velocidade do gás aumenta e o parâmetro de capacidade $Y \propto v_G^2$ se eleva, fazendo o ponto subir em direção à curva vermelha de 100% de inundação. Isso indica que a coluna aproxima-se do limite de arraste e inundação por excesso de velocidade ascendente.

---

### 4. Variação da vazão de líquido ($m_L$) e inundação hidrodinâmica

**Pergunta:**

Se mantivermos $m_G$ constante e aumentarmos a vazão de líquido ($m_L$), o ponto operacional se move para a direita no GPDC. Por que a curva limite de inundação tem inclinação decrescente com o aumento de $X$?

**Resposta esperada:**

À medida que $X$ aumenta (maior fluxo de líquido em relação ao gás), o volume de líquido retido nos interstícios do recheio (*holdup*, $h_L$) aumenta consideravelmente. Havendo mais líquido descendo por gravidade, o espaço livre disponível para o escoamento contínuo do gás diminui. Para evitar que o gás sustente e arraste o líquido, a velocidade máxima admitida de vapor ($v_{Gf}$) tem que ser menor, resultando em uma curva de capacidade máxima admissível ($Y_{\text{flood}}$) estritamente decrescente.

---

### 5. Significado físico do parâmetro de fluxo ($X$ ou $F_{LV}$)

**Pergunta:**

O parâmetro de fluxo $X = (m_L/m_G)\sqrt{\rho_G/\rho_L}$ é adimensional e depende apenas das vazões e das massas específicas. Por que ele não depende das dimensões geométricas da torre (como $D$ ou altura $Z$)?

**Resposta esperada:**

Porque $X$ representa a razão de momento linear / inércia entre a fase líquida descendente e a fase gasosa ascendente. Como a área transversal da coluna ($A$) é a mesma para ambas as fases que escoam em contracorrente, as vazões mássicas superficiais dividem-se pela mesma área ($L'/V' = m_L/m_G$), tornando $X$ uma propriedade puramente hidrodinâmica e operacional do par de correntes em contato no fundo da coluna.

---

### 6. Relação entre a fração de inundação ($f$) e a perda de carga

**Pergunta:**

No painel, ajuste o slider da fração de inundação de projeto para $f = 0{,}70$ (70% do flooding) e anote o diâmetro calculado ($0{,}673\text{ m}$). Em seguida, verifique a perda de carga estimada correspondente a esse diâmetro. Ela coincide com os $300\text{ Pa/m}$ estipulados no projeto original? O que isso nos ensina sobre os critérios de projeto?

**Resposta esperada:**

Não coincide. Para $D = 0{,}673\text{ m}$ ($f = 0{,}70$), a perda de carga situa-se acima de $450\text{ Pa/m}$, ultrapassando o valor de $300\text{ Pa/m}$. Isso demonstra que dimensionar simplesmente por uma regra empírica arbitrária (como "adotar 70% ou 80% do flooding") pode violar restrições estritas de projeto de perda de carga. O critério de $\Delta P = 300\text{ Pa/m}$ neste caso específico exigiu operar mais próximo de $\approx 58\%$ do flooding ($D = 0{,}737\text{ m}$).

---

### 7. Riscos do superdimensionamento da coluna

**Pergunta:**

O que acontece com a coluna se o aluno selecionar um diâmetro excessivamente grande (por exemplo, $D = 1{,}20\text{ m}$)? Por que uma perda de carga excessivamente baixa (ex.: $< 50\text{ Pa/m}$) não é desejável na prática industrial?

**Resposta esperada:**

Embora o risco de inundação seja eliminado e a perda de carga caia para valores quase nulos, surgem três graves problemas operacionais e econômicos:

1. **Má distribuição de líquido (*channeling*):** com baixas velocidades superficiais, o líquido tende a escorrer em canais preferenciais ou pelas paredes da torre, deixando grandes partes do recheio secas e diminuindo drasticamente a área interfacial efetiva de transferência de massa ($a_h$);


2. **Baixa turbulência:** reduz os coeficientes convectivos de transferência de massa ($k_G$ e $k_L$);
3. **Custo de capital elevado (Capex):** colunas com diâmetros desnecessariamente grandes exigem mais aço/material de construção, carcaças mais caras e fundações mais robustas.

---

### 8. Interpretação da faixa recomendada de perda de carga ($200\text{ a }400\text{ Pa/m}$)

**Pergunta:**

Por que na literatura de operações unitárias a faixa entre $200$ e $400\text{ Pa/m}$ (destacada em verde no gráfico da direita) é apontada como a faixa ótima de projeto para torres de absorção?

**Resposta esperada:**

Essa faixa representa o compromisso ideal entre **tamanho da coluna (custo de capital)**, **custo operacional com ventilação (potência do soprador)** e **eficiência de separação**:

* Abaixo de $200\text{ Pa/m}$, a coluna torna-se volumosa e há risco de má distribuição do solvente;
* Acima de $400\text{ a }500\text{ Pa/m}$, o leito entra na região de carregamento (*loading region*), onde a retenção de líquido dispara rapidamente, aproximando a operação do afogamento iminente e elevando de forma proibitiva o consumo elétrico do soprador/compressor de gás de alimentação.



---

### 9. Comparação de desempenho entre anéis Raschig e anéis Hiflow

**Pergunta:**

Compare os *Anéis Raschig 50 mm* com os *Anéis Hiflow 50 mm* nas mesmas condições operacionais ($m_G = 2{,}202\text{ kg/s}$, $m_L = 0{,}804\text{ kg/s}$). Qual dos dois requer um diâmetro menor para manter os mesmos $300\text{ Pa/m}$? Como a evolução histórica do recheio explica essa diferença?

**Resposta esperada:**

Os **Anéis Hiflow 50 mm** permitem um diâmetro significativamente menor ($D \approx 0{,}737\text{ m}$) do que os anéis Raschig de 50 mm.

*Explicação:* Os anéis Raschig são cilindros ocos simples (1ª geração de recheios randômicos) com paredes sólidas que bloqueiam a passagem do gás. Os anéis Hiflow (3ª geração) possuem geometria aberta em treliça/aros vazados, oferecendo porosidade de até $97{,}7\%$ ($\varepsilon = 0{,}977$). Essa estrutura aberta permite a drenagem desimpedida do líquido e a passagem do vapor com menor atrito, conferindo quase o dobro de capacidade de fluxo para um mesmo diâmetro de coluna.

---

### 10. Limitações de extrapolação do modelo hidráulico

**Pergunta:**

O simulador utiliza os modelos de GPDC e Billet & Schultes para estimar as curvas e o ponto operacional. Em quais condições operacionais reais os resultados desse simulador deixariam de ser confiáveis e subestimariam a perda de carga real?

**Resposta esperada:**

Os resultados perderiam a precisão em:

1. **Sistemas espumantes (*foaming*):** como absorção de gases com soluções de aminas, onde a formação de espuma eleva drasticamente a perda de carga e antecipa o afogamento (seria necessário incluir um *Foaming Factor* $FF < 1$ no cálculo);
2. **Pressões operacionais elevadas ($P > 10\text{ bar}$):** onde a densidade do vapor cresce e ocorrem fenômenos de recirculação e arraste não previstos pelas correlações calibradas à pressão atmosférica;


3. **Líquidos altamente viscosos ($\mu_L > 25\text{ cP}$):** onde a hipótese de filme fino e drenagem laminar por gravidade de Billet & Schultes perde validade;


4. **Razão $D / d_p < 8$:** em diâmetros muito pequenos, onde o escoamento preferencial pelas paredes (*efeito de parede*) distorce completamente o perfil de vazios radial da coluna.